# Individual Contribution – Nour Elhefnawy

Datasets: Affordable Housing Production by Building and SIRTA serious crashes (the cleaned CSVs from the main notebook).

The first member used borough totals and monthly trends. I used **location** instead: for every housing building I counted how many serious crashes happened within 400 m (about a 5 minute walk).

**Q3:** Are New Construction and Preservation buildings in different crash areas, and does it depend on the borough?
**Q4:** After a new building is completed, do serious crashes nearby go up or down compared to the rest of the city?

In [1]:
import numpy as np, pandas as pd
from sklearn.neighbors import BallTree
from scipy import stats
import plotly.express as px, plotly.graph_objects as go

# change these paths if running in Colab
housing = pd.read_csv('datasets/Housing_Cleaned.csv', low_memory=False)
crashes = pd.read_csv('datasets/Transportation_Cleaned.csv', low_memory=False)

RADIUS_M = 400
EARTH_R = 6_371_000
BORO_ORDER = ['MANHATTAN', 'BRONX', 'BROOKLYN', 'QUEENS', 'STATEN ISLAND']
print(housing.shape, crashes.shape)

(9551, 41) (13754, 23)


## Extra cleaning

While checking the cleaned files I found a few problems:
- 441 rows in the transportation file are completely empty, so I dropped them.
- `crash_time` has a wrong date (2026-11-08) attached, so I only use `crash_date`.
- Many crashes have no latitude/longitude (about 18%). In Aug–Sep 2024 almost all of them are missing (chart below).
- `community_board` and `council_district` are missing for about 72% of crashes, so I joined using latitude/longitude instead, which keeps more data.

In [2]:
crashes = crashes.dropna(subset=['collision_id']).copy()
crashes['crash_date'] = pd.to_datetime(crashes['crash_date'])
crashes['serious_harm'] = crashes['total_severe_injuries'] + crashes['total_fatalities']

# % of crashes with no coordinates in each month
missing_geo = (crashes.assign(month=crashes.crash_date.dt.to_period('M').astype(str))
               .groupby('month')['latitude'].apply(lambda s: s.isna().mean() * 100)
               .reset_index(name='pct_missing'))
fig = px.bar(missing_geo, x='month', y='pct_missing',
             title='Crashes with no coordinates by month',
             labels={'month': 'Month', 'pct_missing': '% missing lat/long'})
fig.update_layout(template='plotly_white'); fig.show()

# keep only rows that have coordinates
crashes_geo = crashes.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
housing_geo = housing.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
print(len(crashes_geo), 'crashes and', len(housing_geo), 'buildings kept')

10868 crashes and 7593 buildings kept


The missing months hit every building the same way, so I keep them for RQ3. For RQ4 they could cause problems, so I compare against the citywide trend there.

## Integration

Both datasets have latitude/longitude, so I joined them by distance. A BallTree finds all crashes within 400 m of each building, then I count them. This gives one row per building.

In [3]:
crash_tree = BallTree(np.radians(crashes_geo[['latitude', 'longitude']]), metric='haversine')
radius_rad = RADIUS_M / EARTH_R

# list of nearby crashes for each building
neighbours = crash_tree.query_radius(np.radians(housing_geo[['latitude', 'longitude']]), r=radius_rad)

housing_geo['crashes_within_400m'] = [len(idx) for idx in neighbours]
housing_geo['serious_harm_within_400m'] = [crashes_geo['serious_harm'].values[idx].sum() for idx in neighbours]

# check after joining: no missing values
print(housing_geo[['crashes_within_400m', 'serious_harm_within_400m']].isna().sum().to_dict())
housing_geo[['crashes_within_400m', 'serious_harm_within_400m']].describe()

{'crashes_within_400m': 0, 'serious_harm_within_400m': 0}


,crashes_within_400m,serious_harm_within_400m
count,7593.000000,7593.000000
mean,15.943764,17.051626
std,9.222337,9.982636
min,0.000000,0.000000
25%,9.000000,10.000000
50%,15.000000,16.000000
75%,21.000000,23.000000
max,54.000000,62.000000


## Q3: New Construction vs Preservation

New construction is often built on open land while preservation keeps older buildings in dense areas, so I wanted to see if they are near different amounts of crashes. I used the average number of crashes within 400 m per building. Error bars are 95% intervals and I used a Mann–Whitney test because the counts are skewed.

In [4]:
rq3 = (housing_geo.groupby(['borough', 'reporting_construction_type'])
       .agg(buildings=('building_id', 'size'),
            mean_crashes=('crashes_within_400m', 'mean'),
            sem=('crashes_within_400m', lambda s: s.std() / np.sqrt(len(s))))
       .reset_index())
rq3['ci95'] = 1.96 * rq3['sem']

fig = px.bar(rq3, x='borough', y='mean_crashes', color='reporting_construction_type', barmode='group',
             error_y='ci95', category_orders={'borough': BORO_ORDER},
             hover_data={'buildings': True},
             labels={'mean_crashes': 'Avg. serious crashes within 400 m',
                     'borough': 'Borough', 'reporting_construction_type': 'Construction type'},
             title='Serious crashes near affordable housing: New Construction vs Preservation')
fig.update_layout(template='plotly_white'); fig.show()

# test each borough
rows = []
for boro in BORO_ORDER:
    d = housing_geo[housing_geo.borough == boro]
    new = d[d.reporting_construction_type == 'New Construction']['crashes_within_400m']
    pres = d[d.reporting_construction_type == 'Preservation']['crashes_within_400m']
    rows.append({'borough': boro, 'n_new': len(new), 'n_preservation': len(pres),
                 'mean_new': new.mean(), 'mean_preservation': pres.mean(),
                 'p_value': stats.mannwhitneyu(new, pres).pvalue})
pd.DataFrame(rows).round(4)

,borough,n_new,n_preservation,mean_new,mean_preservation,p_value
0,MANHATTAN,287,1112,23.8432,21.3103,0.0000
1,BRONX,1004,1185,19.5737,19.1139,0.2890
2,BROOKLYN,1767,1326,13.8930,13.3808,0.0341
3,QUEENS,462,321,9.5173,4.0748,0.0000
4,STATEN ISLAND,35,94,3.1143,1.2021,0.0803


**What I found**
- Queens has the biggest difference: new buildings have about 9.5 crashes nearby and preservation buildings about 4.1.
- Manhattan is also higher for new construction (23.8 vs 21.3).
- Bronx has no clear difference and Brooklyn only a small one.
- Staten Island only has 35 new buildings, so I can't say much.
- The borough matters much more than the construction type. Staten Island buildings have 1–3 crashes nearby while Manhattan has 21–24.

**Limits:** SIRTA only has serious crashes, and I didn't adjust for traffic or population, so busier areas will naturally have more. This shows a relationship, not a cause.

## Q4: Do crashes change after a new building opens?

I took New Construction buildings completed between Jan 2023 and Jun 2025 so each has a full year before and after. For each one I counted crashes within 400 m in the 365 days before and after completion.

Crashes change across the whole city too, so I used the citywide change over the same dates to work out what I would *expect* after. Ratio = actual after ÷ expected after. A ratio of 1 means no change. Confidence intervals use bootstrapping.

In [5]:
new_buildings = housing_geo[housing_geo.reporting_construction_type == 'New Construction'].copy()

# use building completion date, or project completion date if it is missing
new_buildings['completion'] = pd.to_datetime(
    new_buildings['building_completion_date'].fillna(new_buildings['project_completion_date']), errors='coerce')
new_buildings = new_buildings.dropna(subset=['completion'])
new_buildings = new_buildings[(new_buildings.completion >= '2023-01-01') &
                              (new_buildings.completion <= '2025-06-30')].reset_index(drop=True)

nearby = crash_tree.query_radius(np.radians(new_buildings[['latitude', 'longitude']]), r=radius_rad)
crash_dates = crashes_geo['crash_date'].values
sorted_city_dates = np.sort(crash_dates)
window = np.timedelta64(365, 'D')

pre, post, city_pre, city_post = [], [], [], []
for completion, idx in zip(new_buildings['completion'].values, nearby):
    d = crash_dates[idx]
    pre.append(((d >= completion - window) & (d < completion)).sum())
    post.append(((d >= completion) & (d < completion + window)).sum())
    # same two windows but for the whole city
    city_pre.append(np.searchsorted(sorted_city_dates, completion) - np.searchsorted(sorted_city_dates, completion - window))
    city_post.append(np.searchsorted(sorted_city_dates, completion + window) - np.searchsorted(sorted_city_dates, completion))

new_buildings[['pre', 'post', 'city_pre', 'city_post']] = np.column_stack([pre, post, city_pre, city_post])
new_buildings['expected_post'] = new_buildings['pre'] * new_buildings['city_post'] / new_buildings['city_pre']

def observed_over_expected(d):
    return d['post'].sum() / d['expected_post'].sum()

def bootstrap_ci(d, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    vals = [observed_over_expected(d.sample(len(d), replace=True, random_state=int(rng.integers(1e9))))
            for _ in range(n_boot)]
    return np.percentile(vals, [2.5, 97.5])

lo, hi = bootstrap_ci(new_buildings)
rows = [{'group': 'All NYC', 'buildings': len(new_buildings),
         'ratio': observed_over_expected(new_buildings), 'low': lo, 'high': hi}]
# Staten Island left out, only 4 buildings
for boro in ['MANHATTAN', 'BRONX', 'BROOKLYN', 'QUEENS']:
    d = new_buildings[new_buildings.borough == boro]
    lo, hi = bootstrap_ci(d, n_boot=1000)
    rows.append({'group': boro, 'buildings': len(d), 'ratio': observed_over_expected(d), 'low': lo, 'high': hi})
rq4 = pd.DataFrame(rows)
rq4

,group,buildings,ratio,low,high
0,All NYC,926,0.949701,0.901900,0.998241
1,MANHATTAN,55,1.044530,0.904476,1.218749
2,BRONX,317,0.940581,0.870867,1.018927
3,BROOKLYN,409,0.963926,0.890465,1.038980
4,QUEENS,141,0.862866,0.728432,1.026824


In [6]:
fig = go.Figure(go.Bar(
    x=rq4['group'], y=rq4['ratio'] - 1, base=1,
    error_y=dict(type='data', symmetric=False, array=rq4['high'] - rq4['ratio'], arrayminus=rq4['ratio'] - rq4['low']),
    marker_color=['#444444'] + ['#1f77b4'] * 4,
    customdata=rq4[['buildings', 'low', 'high']],
    hovertemplate='%{x}<br>Ratio: %{y:.2f}<br>Buildings: %{customdata[0]}<br>95% CI: %{customdata[1]:.2f} to %{customdata[2]:.2f}<extra></extra>'))
fig.add_hline(y=1, line_dash='dash', annotation_text='No change')
fig.update_yaxes(title='Crashes after / expected crashes after')
fig.update_layout(template='plotly_white',
                  title='Serious crashes within 400 m after a new building opens vs citywide trend')
fig.show()

In [7]:
# check: only keep buildings whose windows don't overlap the Jul-Sep 2024 missing coordinates
gap_start, gap_end = pd.Timestamp('2024-07-01'), pd.Timestamp('2024-09-30')
overlaps_gap = ((new_buildings.completion - pd.Timedelta(days=365) <= gap_end) &
                (new_buildings.completion + pd.Timedelta(days=365) >= gap_start))
no_gap = new_buildings[~overlaps_gap]
print(len(no_gap), 'of', len(new_buildings), 'buildings kept')
print('Ratio:', round(observed_over_expected(no_gap), 3), '| 95% CI:', bootstrap_ci(no_gap).round(3))

160 of 926 buildings kept
Ratio: 1.018 | 95% CI: [0.906 1.145]


**What I found**
- For all 926 new buildings the ratio is about 0.95 (interval 0.90 to 1.00), so crashes were slightly lower than expected.
- Queens is lowest (about 0.86) and Manhattan is the only one above 1 (about 1.05), but their intervals include 1.
- When I only kept the 160 buildings that avoid the missing months, the ratio was about 1.02, so the small drop is not reliable.

So there is no clear sign that serious crashes go up after a new affordable building opens.

**Limits:** one year is short and each building only has about 3–4 crashes nearby, so results are noisy. Other things (street redesigns, speed cameras) could also change crashes, and about 18% of crashes are missing coordinates. This does not show the building causes any change.

**Benefit for NYC:** planners can focus street-safety work on the high-crash areas from RQ3 instead of assuming new buildings make streets more dangerous.

## My contribution
- Research questions RQ3 and RQ4 with one chart each
- Extra cleaning checks (empty rows, crash_time, missing coordinates)
- Distance-based integration of the two datasets
- Before/after comparison with a citywide control